In [ ]:
# Código para Regressão simples
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan

#Leitura do arquivo excel
df = pd.read_excel('/content/sample_data/supermercadoimperio_semoutliers.xlsx')
print(df.head())

#Gráfico de dispersão
sns.lmplot(x='idade', y='salario', data=df)
plt.title('Dispersão entre Idade e Salário')
plt.xlabel('Idade')
plt.ylabel('Salário')
plt.show()


# Define as variáveis dependente e independente
X = df['idade']
y = df['salario']

# Adicione a constante ao modelo (b0)
X = sm.add_constant(X)

# ajuste o modelo de regressão simples
model = sm.OLS(y, X).fit()

# mostra resumo dos resultados
print(model.summary())


y_pred = model.predict(X)

# MAPE
mape = np.mean(np.abs((y - y_pred) / y)) * 100

print(f'MAPE: {mape:.2f}%')

# Previsão para o salário de um indivíduo com 40 anos e outro com 50 anos
new_ages = pd.DataFrame({'const': 1, 'idade': [40, 50]})
prediction_summary = model.get_prediction(new_ages)

prediction_intervals = prediction_summary.summary_frame(alpha=0.05)

print("Intervalo de predição de 95% para o salário de um funcionário de 40 anos:")
print(f"R$ {prediction_intervals['obs_ci_lower'][0]:.2f} - R$ {prediction_intervals['obs_ci_upper'][0]:.2f}")

print("\nIntervalo de predição de 95% para o salário de um funcionário de 50 anos:")
print(f"R$ {prediction_intervals['obs_ci_lower'][1]:.2f} - R$ {prediction_intervals['obs_ci_upper'][1]:.2f}")

# Gráfico de dispersão com intervalos de confiança e predição
sns.lmplot(x='idade', y='salario', data=df, ci=None, height=6, aspect=1.5)

prediction_summary_all = model.get_prediction(X)
prediction_intervals_all = prediction_summary_all.summary_frame(alpha=0.05)


plt.plot(df['idade'], prediction_intervals_all['mean_ci_lower'], color='blue', linestyle='--', label='Intervalo de Confiança')
plt.plot(df['idade'], prediction_intervals_all['mean_ci_upper'], color='blue', linestyle='--')

plt.plot(df['idade'], prediction_intervals_all['obs_ci_lower'], color='red', linestyle='--', label='Intervalo de Predição')
plt.plot(df['idade'], prediction_intervals_all['obs_ci_upper'], color='red', linestyle='--')


plt.title('Dispersão entre Idade e Salário com Intervalos de Confiança e Predição')
plt.xlabel('Idade')
plt.ylabel('Salário')
plt.legend()
plt.show()
#Análise de resíduos

# Durbin-Watson
dw_statistic = durbin_watson(model.resid)
print(f'Estatística de Durbin-Watson: {dw_statistic:.2f}')

# Shapiro-Wilk
shapiro_statistic, shapiro_pvalue = shapiro(model.resid)
print(f'\nEstatística de Shapiro-Wilk: {shapiro_statistic:.3f}')
print(f'P-valor de Shapiro-Wilk: {shapiro_pvalue:.3f}')

# Breusch-Pagan
bp_test = het_breuschpagan(model.resid, model.model.exog)
labels = ['Estatística LM', 'P-valor LM', 'Estatística F', 'P-valor F']
print('\nTeste de Breusch-Pagan:')
for name, val in zip(labels, bp_test):
    print(f'{name}: {val:.3f}')

# Cálculo dos resíduos padronizados
standardized_residuals = model.get_influence().resid_studentized_internal

# Gráfico de resíduos vs. valores ajustados
plt.scatter(model.fittedvalues, standardized_residuals)
plt.xlabel('Valores Ajustados')
plt.ylabel('Resíduos Padronizados')
plt.title('Resíduos Padronizados vs. Valores Ajustados')
plt.axhline(y=0, color='r', linestyle='--')
plt.show()

# Histograma dos resíduos
sns.histplot(model.resid, kde=True)
plt.xlabel('Resíduos')
plt.title('Histograma dos Resíduos')
plt.show()

In [ ]:
# Código para Regressão com outlier
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan

#Leitura do arquivo excel com outlier
df2 = pd.read_excel('/content/sample_data/supermercadoimperio.xlsx')
print(df2.head())

#Gráfico de dispersão
sns.lmplot(x='idade', y='salario', data=df2)
plt.title('Dispersão entre Idade e Salário')
plt.xlabel('Idade')
plt.ylabel('Salário')
plt.show()

# Define as variáveis dependente e independente
X = df2['idade']
y = df2['salario']

# Adicione a constante ao modelo (b0)
X = sm.add_constant(X)

# ajuste o modelo de regressão simples
model2 = sm.OLS(y, X).fit()

# mostra resumo dos resultados
print(model2.summary())

residuals = model2.resid
fitted_value = model2.fittedvalues
stand_resids = model2.resid_pearson
influence = model2.get_influence()
leverage = influence.hat_matrix_diag

# Gráficos de diagnóstico
plt.rcParams["figure.figsize"] = (20,15)
fig, ax = plt.subplots(nrows=2, ncols=2)

sns.set_style("whitegrid")

# Residuos vs Valores Ajustados
sns.scatterplot(x=fitted_value, y=residuals, ax=ax[0, 0])
ax[0, 0].axhline(y=0, color='grey', linestyle='dashed')
ax[0, 0].set_xlabel('Valores Ajustados')
ax[0, 0].set_ylabel('Residuos')
ax[0, 0].set_title('Residuo vs Valores Ajustados')

# Normal Q-Q
sm.qqplot(residuals, fit=True, line='45',ax=ax[0, 1])
ax[0, 1].set_title('Normal Q-Q')

# Scale-Location
sns.scatterplot(x=fitted_value, y=np.sqrt(np.abs(stand_resids)), ax=ax[1, 0])
ax[1, 0].axhline(y=0, color='grey', linestyle='dashed')
ax[1, 0].set_xlabel('Fitted values')
ax[1, 0].set_ylabel('Sqrt(standardized residuals)')
ax[1, 0].set_title('Scale-Location')

# Residuo vs Leverage
sns.scatterplot(x=leverage, y=stand_resids, ax=ax[1, 1])
ax[1, 1].axhline(y=0, color='grey', linestyle='dashed')
ax[1, 1].set_xlabel('Leverage')
ax[1, 1].set_ylabel('Resíduos padronizados')
ax[1, 1].set_title('Residuos vs Leverage')


plt.tight_layout()
plt.show()



In [ ]:
# Código para Regressão Múltipla

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan

#Leitura do arquivo excel
df = pd.read_excel('/content/sample_data/supermercadoimperio_semoutliers.xlsx')
print(df.head())

# Define as variáveis independentes e dependente
X = df[['idade', 'tempocasa']]
y = df['salario']

# Adiciona a constante ao modelo
X = sm.add_constant(X)

multiple_model = sm.OLS(y, X).fit()
print(multiple_model.summary())

# Cria um dataframe com os valores das variáveis para predição
new_data = pd.DataFrame({'const': 1, 'idade': [40], 'tempocasa': [10]})

predicted_salary = multiple_model.predict(new_data)

print(f'O salário previsto para uma pessoa com 40 anos e 10 anos de casa é de R$ {predicted_salary[0]:.2f}')

prediction_summary = multiple_model.get_prediction(new_data)

prediction_interval = prediction_summary.summary_frame(alpha=0.05)

print("Intervalo de predição de 95% para o salário:")
print(f"R$ {prediction_interval['obs_ci_lower'][0]:.2f} - R$ {prediction_interval['obs_ci_upper'][0]:.2f}")

# Criando as variáveis dummy
df_dummies = pd.get_dummies(df, columns=['educacao', 'cargo', 'local'], drop_first=True, dtype=int)

# Definindo as variáveis do modelo
X = df_dummies.drop(['id', 'salario'], axis=1)
y = df_dummies['salario']

# Adiciona constante ao modelo
X = sm.add_constant(X)

full_model = sm.OLS(y, X).fit()

print(full_model.summary())


In [ ]:
# Código para 4.3.5 - Análise dos resíduos
residuals = full_model.resid
fitted_value = full_model.fittedvalues
stand_resids = full_model.resid_pearson
influence = full_model.get_influence()
leverage = influence.hat_matrix_diag

# Gráficos de diagnóstico
plt.rcParams["figure.figsize"] = (20,15)
fig, ax = plt.subplots(nrows=2, ncols=2)

sns.set_style("whitegrid")

# Residuos vs Valores Ajustados
sns.scatterplot(x=fitted_value, y=residuals, ax=ax[0, 0])
ax[0, 0].axhline(y=0, color='grey', linestyle='dashed')
ax[0, 0].set_xlabel('Valores Ajustados')
ax[0, 0].set_ylabel('Residuos')
ax[0, 0].set_title('Residuo vs Valores Ajustados')

# Normal Q-Q
sm.qqplot(residuals, fit=True, line='45',ax=ax[0, 1])
ax[0, 1].set_title('Normal Q-Q')

# Scale-Location
sns.scatterplot(x=fitted_value, y=np.sqrt(np.abs(stand_resids)), ax=ax[1, 0])
ax[1, 0].axhline(y=0, color='grey', linestyle='dashed')
ax[1, 0].set_xlabel('Fitted values')
ax[1, 0].set_ylabel('Sqrt(standardized residuals)')
ax[1, 0].set_title('Scale-Location')

# Residuo vs Leverage
sns.scatterplot(x=leverage, y=stand_resids, ax=ax[1, 1])
ax[1, 1].axhline(y=0, color='grey', linestyle='dashed')
ax[1, 1].set_xlabel('Leverage')
ax[1, 1].set_ylabel('Resíduos padronizados')
ax[1, 1].set_title('Residuos vs Leverage')


plt.tight_layout()
plt.show()

In [ ]:
# Código para 4.3.6 - Multicolinearidade - Cálculo do VIF

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Cálculo do VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print("VIF:")
print(vif_data)

# Função para calcular o GVIF
def gvif(X, exog_idx):
    """
    Calcula o GVIF.
    """
    X_without_const = X.drop('const', axis=1)
    corr_matrix = np.corrcoef(X_without_const.values.T)
    corr_matrix_exog = corr_matrix[exog_idx, :][:, exog_idx]
    corr_matrix_other = np.delete(np.delete(corr_matrix, exog_idx, axis=0), exog_idx, axis=1)
    gvif = np.linalg.det(corr_matrix_exog) * np.linalg.det(corr_matrix_other) / np.linalg.det(corr_matrix)

    return gvif

# Para 'cargo' (DIRETOR, GERENTE)
cargo_indices = [X.drop('const', axis=1).columns.get_loc('cargo_DIRETOR'), X.drop('const', axis=1).columns.get_loc('cargo_GERENTE')]
gvif_cargo = gvif(X, cargo_indices)
print(f"GVIF for 'cargo': {gvif_cargo:.2f}")

# GVIF^(1/(2*Df))
df_cargo = 2
gvif_adjusted_cargo = gvif_cargo**(1 / (2 * df_cargo))
print(f"Adjusted GVIF for 'cargo': {gvif_adjusted_cargo:.2f}")


In [ ]:
# Código para Regressão best subsets
import itertools
import statsmodels.api as sm
import pandas as pd

# Get the list of predictor variables
predictors = X.columns.drop('const')

# Initialize a list to store the results
results = []

# Loop through all possible numbers of predictors
for i in range(1, len(predictors) + 1):
    # Initialize variables to store the best model for this number of predictors
    best_adj_r2 = -1
    best_model_info = {}

    # Loop through all combinations of predictors of size i
    for combo in itertools.combinations(predictors, i):
        # Create the model with the current combination of predictors
        X_subset = X[['const'] + list(combo)]
        model = sm.OLS(y, X_subset).fit()

        # Check if this model is the best for this number of predictors
        if model.rsquared_adj > best_adj_r2:
            best_adj_r2 = model.rsquared_adj
            best_model_info = {
                'n_predictors': i,
                'predictors': ', '.join(combo),
                'rsquared': f'{model.rsquared:.4f}',
                'adj_rsquared': f'{model.rsquared_adj:.4f}',
                'mallows_cp': f'{model.aic - len(y) + 2 * (i + 1):.4f}', # Mallows' Cp
                'aic': f'{model.aic:.4f}'
            }

    # Add the best model for this number of predictors to the results
    results.append(best_model_info)

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Print the results table
print(results_df)

In [ ]:
# Código para Regressão stepwise
import statsmodels.api as sm

def forward_selection(X, y, significance_level=0.05):
    """
    Perform forward stepwise regression.
    """
    initial_features = X.columns.tolist()
    best_features = []
    while (len(initial_features) > 0):
        remaining_features = list(set(initial_features) - set(best_features))
        new_pval = pd.Series(index=remaining_features)
        for new_column in remaining_features:
            model = sm.OLS(y, sm.add_constant(X[best_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if (min_p_value < significance_level):
            best_features.append(new_pval.idxmin())
        else:
            break
    return best_features

best_features_forward = forward_selection(X.drop('const', axis=1), y)
print("Melhores preditoras (Forward Selection):", best_features_forward)


final_model_forward = sm.OLS(y, sm.add_constant(X[best_features_forward])).fit()
print(final_model_forward.summary())

# Predição
new_director_data = pd.DataFrame({
    'const': 1,
    'idade': [30],
    'tempocasa': [5],
    'cargo_DIRETOR': [1],
    'cargo_GERENTE': [0]
})

predicted_salary_director = final_model_forward.predict(new_director_data)

print(f'O salário previsto para um diretor de 30 anos com 5 anos de casa é de R$ {predicted_salary_director[0]:.2f}')

prediction_summary_director = final_model_forward.get_prediction(new_director_data)
prediction_interval_director = prediction_summary_director.summary_frame(alpha=0.05)

print("\nIntervalo de predição de 95% para o salário:")
print(f"R$ {prediction_interval_director['obs_ci_lower'][0]:.2f} - R$ {prediction_interval_director['obs_ci_upper'][0]:.2f}")

# Modelo final e intervalo de predição
X_final = df_dummies[['idade', 'tempocasa', 'cargo_DIRETOR', 'cargo_GERENTE']]
y_final = df_dummies['salario']

X_final = sm.add_constant(X_final)
final_model = sm.OLS(y_final, X_final).fit()
print(final_model.summary())

# Cria dataframe para previsão
new_director_data = pd.DataFrame({
    'const': 1,
    'idade': [30],
    'tempocasa': [5],
    'cargo_DIRETOR': [0],
    'cargo_GERENTE': [1]
})

predicted_salary_director = final_model.predict(new_director_data)
print(f'O salário previsto para um diretor de 30 anos com 5 anos de casa é de R$ {predicted_salary_director[0]:.2f}')
prediction_summary_director = final_model.get_prediction(new_director_data)
prediction_interval_director = prediction_summary_director.summary_frame(alpha=0.05)
print("\nIntervalo de predição de 95% para o salário:")
print(f"R$ {prediction_interval_director['obs_ci_lower'][0]:.2f} - R$ {prediction_interval_director['obs_ci_upper'][0]:.2f}")

In [ ]:
# Código para 4.3.9 - Modelos com interação
import seaborn as sns
import matplotlib.pyplot as plt

# Cria gráfico de dispersão para visualizar interação
sns.lmplot(x='idade', y='salario', hue='local', data=df, aspect=1.5)

plt.title('Dispersão entre Idade e Salário por Local')
plt.xlabel('Idade')
plt.ylabel('Salário')
plt.show()

# Cria o termo de interação
df_dummies['idade_x_local'] = df_dummies['idade'] * df_dummies['local_INTERIOR']

# Define as variáveis do modelo
X_interaction = df_dummies[['idade', 'local_INTERIOR', 'idade_x_local']]
y_interaction = df_dummies['salario']

# Adiciona a constante
X_interaction = sm.add_constant(X_interaction)

interaction_model = sm.OLS(y_interaction, X_interaction).fit()
print(interaction_model.summary())

In [ ]:
# Código para regressão Ridge e Lasso
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Leitura do arquivo excel
df_car = pd.read_excel('/content/sample_data/2005 CAR DATA.xlsx')

# Calcular a matriz de correlação
correlation_matrix = df_car.corr(numeric_only=True)

# Exibir a matriz de correlação
print("Matriz de Correlação:")
display(correlation_matrix)


In [7]:
#Regressão Ridge e Lasso
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import numpy as np

# Define as variáveis dependente e independentes
X = df_car[['Mileage', 'Cylinder', 'Liter', 'Doors', 'Cruise', 'Sound']]
y = df_car['Price']

# Padroniza as variáveis independentes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Divide os dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
#Lista de valores de alpha a percorrer
alphas = np.logspace(-4, 4, 100)
print(alphas[:10])
print(alphas[-10:])

In [9]:
# Faz divisão para validação cruzada
ridge_cv_model = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
lasso_cv_model = LassoCV(alphas=alphas, cv=5)

In [ ]:
ridge_cv_model.fit(X_train, y_train)
lasso_cv_model.fit(X_train, y_train)

print(f'Alpha ótimo - Ridge: {ridge_cv_model.alpha_:.4f}')
print(f'Alpha ótimo - Lasso: {lasso_cv_model.alpha_:.4f}')

In [ ]:
# Treina o modelo Ridge com alpha ótimo
final_ridge_model = Ridge(alpha=ridge_cv_model.alpha_)
final_ridge_model.fit(X_train, y_train)

# Treina o modelo Lasso com alpha ótimo
final_lasso_model = Lasso(alpha=lasso_cv_model.alpha_)
final_lasso_model.fit(X_train, y_train)

In [ ]:
# Avaliação do modelo final
y_pred_ridge_final = final_ridge_model.predict(X_test)
mse_ridge_final = mean_squared_error(y_test, y_pred_ridge_final)
r2_ridge_final = r2_score(y_test, y_pred_ridge_final)

print("Performance do modelo Ridge (nos dados de teste):")
print(f'Erro quadrático médio: {mse_ridge_final:.2f}')
print(f'R-quadrado (R2): {r2_ridge_final:.2f}')


y_pred_lasso_final = final_lasso_model.predict(X_test)
mse_lasso_final = mean_squared_error(y_test, y_pred_lasso_final)
r2_lasso_final = r2_score(y_test, y_pred_lasso_final)

print("\nPerformance do modelo Lasso (nos dados de teste):")
print(f'Erro quadrático médio: {mse_lasso_final:.2f}')
print(f'R-quadrado (R2): {r2_lasso_final:.2f}')

In [ ]:
print(f'Alpha ótimo - Ridge: {ridge_cv_model.alpha_:.4f}')
print(f'Alpha ótimo - Lasso: {lasso_cv_model.alpha_:.4f}')

print('\nCoeficientes do Modelo Ridge Final (Padronizado):')
for feature, coef in zip(X.columns, final_ridge_model.coef_):
    print(f'{feature}: {coef:.2f}')
print(f'Intercepto : {final_ridge_model.intercept_:.2f}')

print('\nCoeficientes do Modelo Lasso Final (Padronizado):')
for feature, coef in zip(X.columns, final_lasso_model.coef_):
    print(f'{feature}: {coef:.2f}')
print(f'Intercepto : {final_lasso_model.intercept_:.2f}')

print("\nPerformance do modelo Ridge (nos dados de teste):")
print(f'Erro quadrático médio: {mse_ridge_final:.2f}')
print(f'R-quadrado (R2): {r2_ridge_final:.2f}')

print("\nPerformance do modelo Lasso (nos dados de teste):")
print(f'Erro quadrático médio: {mse_lasso_final:.2f}')
print(f'R-quadrado (R2): {r2_lasso_final:.2f}')

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score

# OLS com dados padronizados para comparação
X_scaled_const = sm.add_constant(X_scaled)
ols_model = sm.OLS(y, X_scaled_const).fit()

# Métricas do modelo OLS
y_pred_ols = ols_model.predict(sm.add_constant(X_test))
mse_ols = mean_squared_error(y_test, y_pred_ols)
r2_ols = r2_score(y_test, y_pred_ols)

# Tabela de comparação
ols_coef = ols_model.params.drop('const')
ridge_coef = pd.Series(final_ridge_model.coef_, index=X.columns)
lasso_coef = pd.Series(final_lasso_model.coef_, index=X.columns)

data = {
    'Métrica': ['MSE', 'R-quadrado'] + list(X.columns),
    'OLS ': [mse_ols, r2_ols] + ols_coef.tolist(),
    'Ridge ': [mse_ridge_final, r2_ridge_final] + ridge_coef.tolist(),
    'Lasso ': [mse_lasso_final, r2_lasso_final] + lasso_coef.tolist()
}

comparison_df = pd.DataFrame(data)
numeric_cols = ['OLS ', 'Ridge ', 'Lasso ']
for col in numeric_cols:
    comparison_df[col] = comparison_df[col].apply(lambda x: f'{x:.2f}' if isinstance(x, (int, float)) else x)

display(comparison_df)